## Two-Tower 500k — Colab GPU Training

**Before running:**
1. Runtime > Change runtime type > T4 GPU
2. Upload service account JSON when Cell 2 prompts you
3. Expected time: ~40–60 min for 20 epochs on T4


In [ ]:
%%capture
!pip install faiss-gpu google-cloud-bigquery google-cloud-storage \
             gcsfs pyarrow pandas torch --quiet


In [ ]:
from google.colab import files
import os, sys

print("Upload recosys-service-account.json:")
uploaded = files.upload()
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = \
    "/content/recosys-service-account.json"
print("Credentials set.")


In [ ]:
import sys
# Replace YOUR_USERNAME with your actual GitHub username
!git clone https://github.com/YOUR_USERNAME/RecoSys.git /content/RecoSys
%cd /content/RecoSys
sys.path.insert(0, '/content/RecoSys')
!ls


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")


In [ ]:
# Upload the artifacts/500k/ folder contents from your local machine.
# Easiest: zip it locally, upload here, unzip.
# Run this on your LOCAL machine first:
#   cd artifacts && zip -r 500k_artifacts.zip 500k/
# Then upload the zip here:

from google.colab import files
import zipfile

print("Upload 500k_artifacts.zip:")
uploaded = files.upload()
with zipfile.ZipFile("500k_artifacts.zip", "r") as z:
    z.extractall("artifacts/")
print("Artifacts extracted:")
!ls artifacts/500k/


In [ ]:
import pickle
import pandas as pd

ARTIFACTS_DIR = "artifacts/500k/"
GCS_TEST_PATH = "gs://recosys-data-bucket/samples/users_sample_500k/test/"

with open(f"{ARTIFACTS_DIR}vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

items_enc   = pd.read_parquet(f"{ARTIFACTS_DIR}items_encoded.parquet")
users_enc   = pd.read_parquet(f"{ARTIFACTS_DIR}users_encoded.parquet")
train_pairs = pd.read_parquet(f"{ARTIFACTS_DIR}train_pairs.parquet")
test_df     = pd.read_parquet(GCS_TEST_PATH)

print(f"items_encoded : {items_enc.shape}")
print(f"users_encoded : {users_enc.shape}")
print(f"train_pairs   : {train_pairs.shape}")
print(f"test_df       : {test_df.shape}")
print(f"n_users       : {len(vocabs['user2idx']):,}")
print(f"n_items       : {len(vocabs['item2idx']):,}")


## Config
Edit the cell below before training. USE_CONFIDENCE_WEIGHTING=False
based on 50k experiments (see reports/05_model_experiments_50k.md).


In [ ]:
# ── Hyperparameters ──────────────────────────────────────
N_EPOCHS                 = 20
BATCH_SIZE               = 2048
LEARNING_RATE            = 1e-3
WEIGHT_DECAY             = 1e-5
TEMPERATURE              = 0.05
EVAL_EVERY               = 5
USE_CONFIDENCE_WEIGHTING = False
CHECKPOINT_DIR           = "artifacts/500k/checkpoints/"

# Trained items filter — critical, scope FAISS to trained items only
TRAINED_ITEM_IDXS = set(train_pairs.item_idx.unique())
print(f"Trained items : {len(TRAINED_ITEM_IDXS):,} of "
      f"{len(vocabs['item2idx']):,} total")
print(f"Config ready.")


In [ ]:
import functools
from src.two_tower.models.two_tower import UserTower, ItemTower, TwoTowerModel
from src.two_tower.training.train import train
from src.two_tower.evaluation.evaluate import evaluate

user_tower = UserTower(
    n_users    = len(vocabs['user2idx']),
    n_top_cats = len(vocabs['top_cat_vocab']),
)
item_tower = ItemTower(
    n_items  = len(vocabs['item2idx']),
    n_cat_l1 = len(vocabs['cat_l1_vocab']),
    n_cat_l2 = len(vocabs['cat_l2_vocab']),
    n_brands = len(vocabs['brand_vocab']),
)
model = TwoTowerModel(user_tower, item_tower, temperature=TEMPERATURE)
model.model_summary()

eval_callback = functools.partial(
    evaluate,
    items_encoded_df  = items_enc,
    users_encoded_df  = users_enc,
    test_df           = test_df,
    train_pairs_df    = train_pairs,
    vocabs            = vocabs,
    device            = torch.device(device),
    k                 = 10,
    trained_item_idxs = TRAINED_ITEM_IDXS,
)

history = train(
    model                    = model,
    train_pairs_df           = train_pairs,
    users_encoded_df         = users_enc,
    items_encoded_df         = items_enc,
    n_epochs                 = N_EPOCHS,
    batch_size               = BATCH_SIZE,
    learning_rate            = LEARNING_RATE,
    weight_decay             = WEIGHT_DECAY,
    device_str               = device,
    checkpoint_dir           = CHECKPOINT_DIR,
    eval_every               = EVAL_EVERY,
    eval_fn                  = eval_callback,
    use_confidence_weighting = USE_CONFIDENCE_WEIGHTING,
)


In [ ]:
from google.cloud import storage

def upload_to_gcs(local_path, blob_path,
                  bucket_name="recosys-data-bucket"):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob   = bucket.blob(blob_path)
    blob.upload_from_filename(local_path)
    print(f"Uploaded → gs://{bucket_name}/{blob_path}")

# Upload all checkpoints
import os
ckpt_files = sorted(os.listdir(CHECKPOINT_DIR))
for f in ckpt_files:
    upload_to_gcs(
        f"{CHECKPOINT_DIR}{f}",
        f"models/two_tower_500k/{f}"
    )

# Upload artifacts
upload_to_gcs(f"{ARTIFACTS_DIR}vocabs.pkl",
              "models/two_tower_500k/vocabs_500k.pkl")

print("All checkpoints and vocabs saved to GCS.")
print("Training complete — share Recall@10 results.")
